In [2]:
#%pip install requests
#%pip install telethon
#%pip install --upgrade ipykernel
#%pip install requests lxml pandas
#%pip install selenium
#%pip install pymongo

In [3]:
#%pip install requests lxml mysql-connector-python

парсинг сайта с помощью scrapy


In [4]:
#%pip install crochet
#%pip install scrapy
import scrapy
from scrapy.crawler import CrawlerProcess
import crochet

In [5]:
from pathlib import Path

# БАЗА: папка, где лежит ноутбук (переносимо)
BASE_PATH    = Path(".")
PROJECT_ROOT = BASE_PATH / "simple_scrapy_spider"                  # тут scrapy.cfg
SPIDER_DIR   = PROJECT_ROOT / "simple_scrapy_spider" / "spiders"   # тут пауки
SPIDER_FILE  = SPIDER_DIR / "oldgames_catalog.py"
OUTPUT_CSV   = BASE_PATH / "oldgames_dos.csv"

print("BASE_PATH   :", BASE_PATH.resolve())
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("SPIDER_DIR  :", SPIDER_DIR.resolve())
print("SPIDER_FILE :", SPIDER_FILE.resolve())
print("OUTPUT_CSV  :", OUTPUT_CSV.resolve())

# sanity-check
assert (PROJECT_ROOT / "scrapy.cfg").is_file(), "Не найден scrapy.cfg в папке проекта (simple_scrapy_spider)."

BASE_PATH   : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2
PROJECT_ROOT: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider
SPIDER_DIR  : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider\simple_scrapy_spider\spiders
SPIDER_FILE : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\simple_scrapy_spider\simple_scrapy_spider\spiders\oldgames_catalog.py
OUTPUT_CSV  : C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\oldgames_dos.csv


In [ ]:
SPIDER_DIR.mkdir(parents=True, exist_ok=True)

fixed_spider = r'''
import re
import scrapy

class OldGamesSpider(scrapy.Spider):
    name = "oldgames"
    allowed_domains = ["old-games.ru", "www.old-games.ru", "static.old-games.ru"]

    custom_settings = {
        "ROBOTSTXT_OBEY": True,
        "DOWNLOAD_DELAY": 0.5,  # 
        "DEFAULT_REQUEST_HEADERS": {
            "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/120.0.0.0 Safari/537.36 (Scrapy for research/edu)",
        },
        "FEED_EXPORT_ENCODING": "utf-8-sig",
    }

    def __init__(self, max_pages=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.page_counter = 0
        self.max_pages = int(max_pages) if max_pages else None
        self.start_urls = [
            "https://www.old-games.ru/catalog/?platform=1&sort=popularity"
        ]

    def parse(self, response):
        self.page_counter += 1

        # ВАЖНО: используем ТОЛЬКО прямые дети ./td[n] (первый столбец содержит вложенную таблицу)
        rows = response.xpath('//table[contains(@class,"listtable")]/tr[starts-with(@id,"game_")]')
        for row in rows:
            # 1) Название и ссылка
            name_link = row.xpath(
                './td[1]//a[starts-with(@href,"/game/") and '
                'not(contains(@href,"/video/")) and '
                'not(contains(@href,"/screenshots/")) and '
                'not(contains(@href,"/covers/"))][1]'
            )
            name = name_link.xpath('normalize-space(text())').get()
            url  = response.urljoin(name_link.xpath('@href').get())

            # 2) Жанр (может быть несколько)
            genres = row.xpath('./td[2]//a/text()').getall()
            genre = " / ".join([g.strip() for g in genres if g.strip()]) or None

            # 3) Год
            year_text = row.xpath('./td[3]//a/text()').re_first(r'\d{4}')
            year = int(year_text) if year_text else None

            # 4) Платформа (оставим fallback на текст)
            platforms = row.xpath('./td[4]//a/text()').getall()
            platform = " / ".join([p.strip() for p in platforms if p.strip()])
            if not platform:
                platform = row.xpath('normalize-space(./td[4])').get() or None

            # 5) Издатель 
            publishers = row.xpath('./td[5]//a/text()').getall()
            publisher = " / ".join([p.strip() for p in publishers if p.strip()])
            if not publisher:
                publisher = row.xpath('normalize-space(./td[5])').get() or None

            # 6) Оценка: число из title у <img> «… — N из 10»
            rating_title = row.xpath('./td[6]//img/@title').get()
            rating = None
            if rating_title:
                m = re.search(r'(\d+)\s*из\s*10', rating_title)
                if m:
                    rating = int(m.group(1))

            yield {
                "Название":  name or None,
                "Жанр":      genre or None,
                "Год":       year,
                "Платформа": platform or None,
                "Издатель":  publisher or None,
                "Оценка":    rating,
                "Ссылка": url,  
            }

        # Пагинация (rel="next" или кнопка ">")
        if self.max_pages is None or self.page_counter < self.max_pages:
            next_rel = response.xpath(
                '//ul[contains(@class,"pager")]//a[@rel="next"]/@href | '
                '//ul[contains(@class,"pager")]//a[normalize-space(text())=">"]/@href'
            ).get()
            if next_rel:
                yield response.follow(next_rel, callback=self.parse)
'''

SPIDER_FILE.write_text(fixed_spider.strip() + "\n", encoding="utf-8")
print("Паук обновлён:", SPIDER_FILE)


Паук обновлён: simple_scrapy_spider\simple_scrapy_spider\spiders\oldgames_catalog.py


In [7]:
import sys, subprocess

cmd = [
    sys.executable, "-m", "scrapy", "crawl", "oldgames",
    "-O", str(OUTPUT_CSV.resolve()),  # абсолютный путь, собранный из относительного
    "-a", "max_pages=2",              # для теста: соберёт 2 страницы; если убрать параметр — пойдёт по всем
    "-s", "LOG_LEVEL=INFO",
]
print("Запуск:", " ".join(cmd))
res = subprocess.run(cmd, cwd=str(PROJECT_ROOT.resolve()))
print("Exit code:", res.returncode)
print("CSV:", OUTPUT_CSV.resolve(), "=>", OUTPUT_CSV.exists())

Запуск: c:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\.venv\Scripts\python.exe -m scrapy crawl oldgames -O C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\oldgames_dos.csv -a max_pages=2 -s LOG_LEVEL=INFO
Exit code: 0
CSV: C:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\oldgames_dos.csv => True


In [8]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
display(df.head(20))

print("\nПроверки на пустые значения:")
for col in ["Жанр", "Платформа", "Издатель"]:
    print(f"{col}: пустых = {int(df[col].isna().sum())}")

,Название,Жанр,Год,Платформа,Издатель,Оценка,Ссылка
0,WarCraft II: Tides of Darkness,Strategy,1995,DOS,Blizzard Entertainment,10,https://www.old-games.ru/game/73.html
1,Blood,Action,1997,DOS,GT Interactive Software,10,https://www.old-games.ru/game/11.html
2,Quake,Action,1996,DOS,id Software,10,https://www.old-games.ru/game/64.html
3,X-COM: UFO Defense,Strategy,1994,DOS,MicroProse Software,8,https://www.old-games.ru/game/77.html
4,DOOM,Action,1993,DOS,id Software,10,https://www.old-games.ru/game/4788.html
5,Dune II: The Building of a Dynasty,Strategy,1992,DOS,Virgin Games,10,https://www.old-games.ru/game/1482.html
6,Wolfenstein 3D,Action,1992,DOS,Apogee Software,8,https://www.old-games.ru/game/76.html
7,Sid Meier's Civilization,Strategy,1991,DOS,MicroProse Software,10,https://www.old-games.ru/game/12.html
8,Duke Nukem 3D: Atomic Edition,Action,1996,DOS,GT Interactive Software,10,https://www.old-games.ru/game/604.html
9,Prince of Persia,Arcade,1990,DOS,Brøderbund Software,10,https://www.old-games.ru/game/62.html



Проверки на пустые значения:
Жанр: пустых = 0
Платформа: пустых = 0
Издатель: пустых = 0
